# GeometryPredictor — Standalone Channel Geometry from a Trained KAN

This notebook demonstrates how to use `ddr.geometry.GeometryPredictor` to predict
trapezoidal channel geometry (width, depth, area, hydraulic radius, velocity) from
catchment attributes, discharge, and channel slope — **without running the full
routing pipeline**.

The predictor loads a trained KAN checkpoint and computes geometry for any set of
MERIT (or HydroATLAS) reaches in a single call.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import zarr

from ddr.geometry import GeometryPredictor

## 1. Load the GeometryPredictor

`from_checkpoint` loads the trained KAN weights, normalization statistics, and
parameter bounds from a checkpoint + config pair. No routing network is needed.

In [ ]:
EXAMPLE_DIR = Path(".")
DATA_DIR = EXAMPLE_DIR / ".." / ".." / "data"

predictor = GeometryPredictor.from_checkpoint(
    checkpoint_path=EXAMPLE_DIR / "ddr-v0.5.2-merit-geometry-weights.pt",
    config_path=EXAMPLE_DIR / "geometry_config.yaml",
    stats_path=DATA_DIR / "statistics" / "merit_attribute_statistics_merit_global_attributes_v2.nc.json",
    device="cpu",
)

## 2. Load MERIT CONUS reach data

We need three inputs:
- **Catchment attributes** — the 10 KAN input variables from the MERIT attributes NetCDF
- **Discharge** — a representative Q per reach (m³/s)
- **Channel slope** — bed slope per reach (m/m), stored in the adjacency zarr

In [ ]:
# Load all CONUS reach IDs and slope from the adjacency zarr
adjacency = zarr.open_group(DATA_DIR / "merit_conus_adjacency.zarr", mode="r")
comids = adjacency["order"][:]
slope_arr = adjacency["slope"][:]

# Load catchment attributes
attrs_all = xr.open_dataset(DATA_DIR / "merit_global_attributes_v2.nc")

# Select CONUS reaches
attrs = attrs_all.sel(COMID=comids)

# Build slope and discharge DataArrays
slope = xr.DataArray(slope_arr, dims="COMID", coords={"COMID": comids})

# Use upstream area as a proxy for mean annual discharge: Q ~ 0.01 * uparea_km2
# (rough scaling; replace with actual discharge estimates for real applications)
uparea_km2 = 10 ** attrs["log10_uparea"]
discharge = xr.DataArray(
    (0.01 * uparea_km2).values, dims="COMID", coords={"COMID": comids}
)

print(f"Loaded {len(comids):,} CONUS reaches")
print(f"Discharge range: {float(discharge.min()):.3f} – {float(discharge.max()):.1f} m³/s")
print(f"Slope range: {float(slope.min()):.6f} – {float(slope.max()):.4f} m/m")

## 3. Predict geometry

A single call returns an `xr.Dataset` with all 11 geometry variables per reach.

In [ ]:
geometry = predictor.predict(attrs, discharge, slope)
geometry

## 4. Visualize learned parameters and geometry

### 4a. Distributions of learned KAN parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

params = {
    "n (Manning's roughness)": geometry["n"].values,
    "q_spatial (width-depth exponent)": geometry["q_spatial"].values,
    "p_spatial (width coefficient)": geometry["p_spatial"].values,
}

for ax, (label, values) in zip(axes, params.items(), strict=False):
    valid = values[~np.isnan(values)]
    ax.hist(valid, bins=50, edgecolor="black", linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.axvline(np.median(valid), color="red", linestyle="--", label=f"median={np.median(valid):.3f}")
    ax.legend()

fig.suptitle("Learned KAN Parameters across CONUS Reaches", fontsize=13)
fig.tight_layout()
plt.show()

### 4b. Geometry vs. discharge

Show how the predicted geometry scales with discharge across the network.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
q = discharge.values

scatter_vars = [
    ("top_width", "Top Width (m)"),
    ("depth", "Depth (m)"),
    ("velocity", "Velocity (m/s)"),
    ("hydraulic_radius", "Hydraulic Radius (m)"),
]

for ax, (var, ylabel) in zip(axes.flat, scatter_vars, strict=False):
    vals = geometry[var].values
    mask = ~np.isnan(vals)
    ax.scatter(q[mask], vals[mask], s=0.5, alpha=0.2, rasterized=True)
    ax.set_xlabel("Discharge (m³/s)")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(var)

fig.suptitle("Predicted Channel Geometry vs. Discharge", fontsize=13)
fig.tight_layout()
plt.show()

### 4c. Leopold & Maddock power law: Width = p * Depth^q

Verify the learned relationship follows the expected power-law form.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

depth = geometry["depth"].values
width = geometry["top_width"].values
mask = ~(np.isnan(depth) | np.isnan(width))

ax.scatter(depth[mask], width[mask], s=0.5, alpha=0.2, rasterized=True, label="Predicted")
ax.set_xlabel("Depth (m)")
ax.set_ylabel("Top Width (m)")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Leopold & Maddock: Width vs. Depth")

# Overlay median power law: W = p_median * D^q_median
p_med = np.nanmedian(geometry["p_spatial"].values)
q_med = np.nanmedian(geometry["q_spatial"].values)
d_range = np.logspace(np.log10(np.nanmin(depth)), np.log10(np.nanmax(depth)), 100)
ax.plot(d_range, p_med * d_range**q_med, "r-", linewidth=2,
        label=f"W = {p_med:.1f} D^{q_med:.2f} (median params)")
ax.legend()

fig.tight_layout()
plt.show()

## 5. Predict at multiple discharge levels

The predictor can be called repeatedly with different Q values to show how
geometry varies with flow regime for a single reach.

In [ ]:
# Pick a single reach and sweep discharge from 1 to 1000 m³/s
reach_id = comids[1000]
reach_attrs = attrs.sel(COMID=[reach_id])
reach_slope = slope.sel(COMID=[reach_id])

q_values = np.logspace(0, 3, 50)  # 1 to 1000 m³/s
results = []
for q_val in q_values:
    q_da = xr.DataArray([q_val], dims="COMID", coords={"COMID": [reach_id]})
    geo = predictor.predict(reach_attrs, q_da, reach_slope)
    results.append({var: float(geo[var].values[0]) for var in geo.data_vars})

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(q_values, [r["top_width"] for r in results])
axes[0].set_xlabel("Q (m³/s)")
axes[0].set_ylabel("Top Width (m)")
axes[0].set_xscale("log")

axes[1].plot(q_values, [r["depth"] for r in results])
axes[1].set_xlabel("Q (m³/s)")
axes[1].set_ylabel("Depth (m)")
axes[1].set_xscale("log")

axes[2].plot(q_values, [r["velocity"] for r in results])
axes[2].set_xlabel("Q (m³/s)")
axes[2].set_ylabel("Velocity (m/s)")
axes[2].set_xscale("log")

fig.suptitle(f"Geometry vs. Discharge for COMID {reach_id}", fontsize=13)
fig.tight_layout()
plt.show()

## 6. Summary statistics

In [ ]:
import pandas as pd

summary = {}
for var in geometry.data_vars:
    vals = geometry[var].values
    summary[var] = {
        "min": np.nanmin(vals),
        "p10": np.nanpercentile(vals, 10),
        "median": np.nanmedian(vals),
        "mean": np.nanmean(vals),
        "p90": np.nanpercentile(vals, 90),
        "max": np.nanmax(vals),
    }

pd.DataFrame(summary).T.round(4)